# ETL Exploration
Use this notebook to prototype and test your ETL pipeline before moving code to `etl/etl.py`.

In [69]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()
SIMFIN_API_KEY = os.getenv('SIMFIN_API_KEY')
print('API key loaded:', bool(SIMFIN_API_KEY))

API key loaded: False


## 1. Fetch raw data

In [70]:
# TODO
df = pd.read_csv('../data/raw/us-shareprices-daily.csv', sep=';')

## 2. Clean & explore
Check for nulls, data types, date ranges.

In [71]:
df.shape

(6206232, 11)

In [72]:
df.dtypes

Ticker                 object
SimFinId                int64
Date                   object
Open                  float64
High                  float64
Low                   float64
Close                 float64
Adj. Close            float64
Volume                  int64
Dividend              float64
Shares Outstanding    float64
dtype: object

In [73]:
df.isnull().sum()

Ticker                    668
SimFinId                    0
Date                        0
Open                        0
High                        0
Low                         0
Close                       0
Adj. Close                  0
Volume                      0
Dividend              6169933
Shares Outstanding     525204
dtype: int64

### How are we handling missing values?
1. We are going to apply fillna(0) to Dividend
2. Not changing Shares outstanding as we are not going to use it as a feature. 
3. In regards to Ticker, it won't affect anything since we are going to filter by specific tickets.

In [74]:
#Check date range
print(f'Min date: {df.Date.min()}, Max date: {df.Date.max()}')

Min date: 2020-04-06, Max date: 2025-03-10


In [75]:
companies_df = pd.read_csv('../data/raw/us-companies.csv', sep=';')
companies_df.head()

,Ticker,SimFinId,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,Market,CIK,Main Currency
0,NaN,20095034,NaN,NaN,NaN,NaN,NaN,NaN,us,1986247.0,USD
1,NaN,18692750,NaN,NaN,NaN,NaN,NaN,NaN,us,1997711.0,USD
2,NaN,18847915,NaN,NaN,NaN,NaN,NaN,NaN,us,1769731.0,USD
3,NaN,18538670,NaN,NaN,NaN,NaN,NaN,NaN,us,1734107.0,USD
4,NaN,18657366,NaN,NaN,NaN,NaN,NaN,NaN,us,1899830.0,USD


In [76]:
companies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6529 entries, 0 to 6528
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Ticker                         6491 non-null   object 
 1   SimFinId                       6529 non-null   int64  
 2   Company Name                   6494 non-null   object 
 3   IndustryId                     6225 non-null   float64
 4   ISIN                           5345 non-null   object 
 5   End of financial year (month)  6495 non-null   float64
 6   Number Employees               5703 non-null   float64
 7   Business Summary               6234 non-null   object 
 8   Market                         6529 non-null   object 
 9   CIK                            6517 non-null   float64
 10  Main Currency                  6529 non-null   object 
dtypes: float64(4), int64(1), object(6)
memory usage: 561.2+ KB


## Building Data Cleansing Functions

**Why?** We want to avoid redundancy and being more time-efficient

In [77]:
TICKERS = ['AAPL', 'PLTR', 'GOOG', 'AMZN', 'MSFT']

def clean(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['Dividend'] = df['Dividend'].fillna(0)
    df = df.drop(columns=['SimFinId'])

    # Remove outliers: daily returns above 50% are almost certainly data errors
    # or unadjusted stock splits — keeping them would distort all rolling features.
    daily_return = df['Adj. Close'].pct_change()
    df = df[daily_return.abs() < 0.5].reset_index(drop=True)

    return df

## 3. Feature engineering
Create the derived columns the model will use.

In [78]:
def engineer_features(df):
    df = df.copy()
    df['Return'] = df['Close'].pct_change()
    df['MA5'] = df['Close'].rolling(5).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['Volume_Change'] = df['Volume'].pct_change()

    # Market Cap: total market value = price × shares outstanding.
    # Changes daily with price — a dynamic proxy for company size.
    df['Market_Cap'] = df['Close'] * df['Shares Outstanding']

    # Target: next-day return (continuous regression target).
    # pct_change() gives today's return; .shift(-1) moves tomorrow's return to today's row.
    # Result: for each row, Target = (price_tomorrow - price_today) / price_today
    # Positive = price rose next day, negative = price fell.
    df['Target'] = df['Close'].pct_change().shift(-1)

    # RSI (Relative Strength Index): measures momentum --> >70 = overbought, <30 = oversold
    delta = df['Close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss))

    # MACD (Moving Average Convergence Divergence): measures trend direction and momentum
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

    # Bollinger Bands: measures volatility. Price near upper band = overbought, lower = oversold
    std20 = df['Close'].rolling(20).std()
    df['BB_Upper'] = df['MA20'] + 2 * std20
    df['BB_Lower'] = df['MA20'] - 2 * std20

    # Lag features: gives the model memory of recent returns
    df['Return_Lag1'] = df['Return'].shift(1)  # yesterday's return
    df['Return_Lag2'] = df['Return'].shift(2)  # 2 days ago

    df = df.dropna().reset_index(drop=True)
    return df

processed = {}
for ticker in TICKERS:
    company_df = df[df['Ticker'] == ticker]
    cleaned = clean(company_df)
    featured = engineer_features(cleaned)
    featured.to_csv(f'../data/processed/{ticker}.csv', index=False)
    processed[ticker] = featured
    print(f'{ticker}: {len(featured)} rows saved')

AAPL: 1217 rows saved
PLTR: 1094 rows saved
GOOG: 1217 rows saved
AMZN: 1217 rows saved
MSFT: 1217 rows saved


## 4. Fundamental Data Enrichment

We enrich daily price data with quarterly financial ratios from income statements, balance sheets, and cash flow statements.

**Key challenge — look-ahead bias:** quarterly reports are published weeks *after* the fiscal period ends. We use `Publish Date` (not `Fiscal Year`) so we only attach data that was publicly available on each trading day.

**Technique:** `pd.merge_asof` with `direction='backward'` — for each trading day, attach the most recently *published* quarterly snapshot. This guarantees zero look-ahead bias.

| Source | Raw columns used | Derived feature | Meaning |
|---|---|---|---|
| Income | Gross Profit / Revenue | `Gross_Margin` | Operational efficiency |
| Income | Operating Income / Revenue | `Operating_Margin` | Core profitability |
| Income | Net Income / Revenue | `Net_Margin` | Bottom-line profitability |
| Balance | Total Liabilities / Total Equity | `Debt_to_Equity` | Financial leverage / risk |
| Cashflow + Balance | Operating CF / Total Assets | `Operating_CF_Ratio` | Cash generation quality |

In [79]:
# Load only the columns we need from each quarterly file
income_q = pd.read_csv('../data/raw/us-income-quarterly.csv', sep=';',
    usecols=['Ticker', 'Publish Date', 'Revenue', 'Gross Profit',
             'Operating Income (Loss)', 'Net Income'])

balance_q = pd.read_csv('../data/raw/us-balance-quarterly.csv', sep=';',
    usecols=['Ticker', 'Publish Date', 'Total Assets', 'Total Liabilities', 'Total Equity'])

cashflow_q = pd.read_csv('../data/raw/us-cashflow-quarterly.csv', sep=';',
    usecols=['Ticker', 'Publish Date', 'Net Cash from Operating Activities'])

# Filter to our 5 tickers only
income_q   = income_q[income_q['Ticker'].isin(TICKERS)].copy()
balance_q  = balance_q[balance_q['Ticker'].isin(TICKERS)].copy()
cashflow_q = cashflow_q[cashflow_q['Ticker'].isin(TICKERS)].copy()

for name, frame in [('Income', income_q), ('Balance', balance_q), ('Cashflow', cashflow_q)]:
    n_quarters = frame[frame['Ticker'] == 'AAPL']['Publish Date'].nunique()
    print(f"{name:10s} — {len(frame):4d} rows | AAPL: {n_quarters} quarters")

print()
income_q[income_q['Ticker'] == 'AAPL'][['Ticker', 'Publish Date', 'Revenue', 'Gross Profit', 'Net Income']].tail(4)

Income     —   94 rows | AAPL: 19 quarters
Balance    —   94 rows | AAPL: 19 quarters
Cashflow   —   94 rows | AAPL: 19 quarters



,Ticker,Publish Date,Revenue,Gross Profit,Net Income
142,AAPL,2024-05-03,9.075300e+10,4.227100e+10,23636000000
143,AAPL,2024-08-02,8.577700e+10,3.967800e+10,21448000000
144,AAPL,2024-11-01,9.493000e+10,4.387900e+10,14736000000
145,AAPL,2025-01-31,1.243000e+11,5.827500e+10,36330000000


In [80]:
FUNDAMENTAL_FEATURES = ['Gross_Margin', 'Operating_Margin', 'Net_Margin',
                        'Debt_to_Equity', 'Operating_CF_Ratio']

def compute_fundamental_features(income, balance, cashflow):
    """
    Compute normalized financial ratios from raw quarterly statements.
    Dimensionless ratios are comparable across companies of different sizes.
    """
    # --- Income ratios ---
    inc = income.copy()
    inc['Gross_Margin']     = inc['Gross Profit'] / inc['Revenue']
    inc['Operating_Margin'] = inc['Operating Income (Loss)'] / inc['Revenue']
    inc['Net_Margin']       = inc['Net Income'] / inc['Revenue']
    inc = inc[['Ticker', 'Publish Date', 'Gross_Margin', 'Operating_Margin', 'Net_Margin']]

    # --- Balance ratio ---
    bal = balance.copy()
    # Debt-to-Equity: how much of the company is financed by debt vs equity.
    # High D/E = higher financial risk. Can be negative if equity < 0 (distressed).
    bal['Debt_to_Equity'] = bal['Total Liabilities'] / bal['Total Equity']
    bal_ratios = bal[['Ticker', 'Publish Date', 'Debt_to_Equity']]

    # --- Cash flow ratio (normalised by total assets) ---
    # Operating CF Ratio: cash generated from core operations relative to asset base.
    # More reliable signal than net income — harder to manipulate with accounting.
    cf = cashflow.merge(
        bal[['Ticker', 'Publish Date', 'Total Assets']],
        on=['Ticker', 'Publish Date'], how='left'
    )
    cf['Operating_CF_Ratio'] = cf['Net Cash from Operating Activities'] / cf['Total Assets']
    cf_ratios = cf[['Ticker', 'Publish Date', 'Operating_CF_Ratio']]

    # --- Combine all ---
    fund = inc.merge(bal_ratios, on=['Ticker', 'Publish Date'], how='outer')
    fund = fund.merge(cf_ratios,  on=['Ticker', 'Publish Date'], how='outer')
    fund['Publish Date'] = pd.to_datetime(fund['Publish Date'])
    # Replace inf (e.g. Revenue=0, or Equity=0) with NaN so dropna handles it cleanly
    fund = fund.replace([np.inf, -np.inf], np.nan)
    fund = fund.sort_values(['Ticker', 'Publish Date']).reset_index(drop=True)
    return fund

fundamentals = compute_fundamental_features(income_q, balance_q, cashflow_q)

print(f"Fundamentals shape: {fundamentals.shape}")
print(f"\nSample — AAPL recent quarters:")
fundamentals[fundamentals['Ticker'] == 'AAPL'][['Publish Date'] + FUNDAMENTAL_FEATURES].tail(6)

Fundamentals shape: (98, 7)

Sample — AAPL recent quarters:


,Publish Date,Gross_Margin,Operating_Margin,Net_Margin,Debt_to_Equity,Operating_CF_Ratio
13,2023-11-03,0.451708,0.301336,0.256497,4.673462,0.061256
14,2024-02-02,0.458750,0.337637,0.283638,3.770769,0.112853
15,2024-05-03,0.465781,0.307428,0.260443,3.547686,0.067247
16,2024-08-02,0.462572,0.295557,0.250044,3.971098,0.087023
17,2024-11-01,0.462225,0.311714,0.155230,5.408780,0.073459
18,2025-01-31,0.468825,0.344586,0.292277,4.154214,0.086999


In [81]:
def merge_fundamentals_for_ticker(price_df, fundamentals, ticker):
    """
    Point-in-time merge for a single ticker.

    For each daily trading row, attaches the most recently PUBLISHED quarterly data.
    merge_asof with direction='backward' ensures we never peek into the future:
    if today is 2024-02-01 and the last earnings were published 2024-01-30, those
    values are used. The next quarter's report (e.g. 2024-04-30) is not visible yet.
    """
    p = price_df.sort_values('Date')
    f = (fundamentals[fundamentals['Ticker'] == ticker]
         .drop(columns='Ticker')
         .sort_values('Publish Date'))

    return pd.merge_asof(
        p, f,
        left_on='Date',
        right_on='Publish Date',
        direction='backward'   # only use data published ON OR BEFORE the trading day
    )


# Re-run the full pipeline: clean → merge fundamentals → engineer features → save
PRICE_FEATURES = ['Return', 'MA5', 'MA20', 'Volume_Change', 'Market_Cap',
                  'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
                  'Return_Lag1', 'Return_Lag2']
ALL_FEATURES = PRICE_FEATURES + FUNDAMENTAL_FEATURES

enriched = {}
for ticker in TICKERS:
    raw     = df[df['Ticker'] == ticker]
    cleaned = clean(raw)
    merged  = merge_fundamentals_for_ticker(cleaned, fundamentals, ticker)
    featured = engineer_features(merged)
    featured.to_csv(f'../data/processed/{ticker}.csv', index=False)
    enriched[ticker] = featured
    n_nan_fund = featured[FUNDAMENTAL_FEATURES].isna().sum().sum()
    print(f"{ticker}: {len(featured)} rows saved | NaN in fundamental features: {n_nan_fund}")

AAPL: 1156 rows saved | NaN in fundamental features: 0
PLTR: 1012 rows saved | NaN in fundamental features: 0
GOOG: 1156 rows saved | NaN in fundamental features: 0
AMZN: 965 rows saved | NaN in fundamental features: 0
MSFT: 1157 rows saved | NaN in fundamental features: 0


In [82]:
# Complete feature inventory — what the ML model will see
sample = enriched['AAPL']

print("=" * 60)
print("COMPLETE FEATURE SET  (AAPL sample)")
print("=" * 60)
print(f"\n{'#':<4} {'Feature':<22} {'Type':<8} {'Min':>10}  {'Max':>10}  {'NaN':>5}")
print("-" * 60)

all_cols = ALL_FEATURES + ['Target']
for i, col in enumerate(all_cols, 1):
    s = sample[col]
    tag = "PRICE" if col in PRICE_FEATURES else ("FUND" if col in FUNDAMENTAL_FEATURES else "TARGET")
    print(f"{i:<4} {col:<22} {tag:<8} {s.min():>10.4f}  {s.max():>10.4f}  {s.isna().sum():>5}")

print(f"\nTotal trainable features : {len(ALL_FEATURES)}  ({len(PRICE_FEATURES)} price + {len(FUNDAMENTAL_FEATURES)} fundamental)")
print(f"Date range               : {sample['Date'].min().date()} → {sample['Date'].max().date()}")
print(f"Total rows (AAPL)        : {len(sample)}")
print(f"\nTarget (next-day return) : mean={sample['Target'].mean():.4f}  std={sample['Target'].std():.4f}"
      f"  range=[{sample['Target'].min():.4f}, {sample['Target'].max():.4f}]")

COMPLETE FEATURE SET  (AAPL sample)

#    Feature                Type            Min         Max    NaN
------------------------------------------------------------
1    Return                 PRICE       -0.0801      0.1047      0
2    MA5                    PRICE       97.1100    256.5140      0
3    MA20                   PRICE       96.0190    249.9850      0
4    Volume_Change          PRICE       -0.8301      3.7720      0
5    Market_Cap             PRICE    1817315475360.0000  3915300473459.9995      0
6    RSI                    PRICE        3.1429     96.1630      0
7    MACD                   PRICE       -6.6731      8.9481      0
8    MACD_Signal            PRICE       -5.8157      8.3104      0
9    BB_Upper               PRICE      101.9048    265.2736      0
10   BB_Lower               PRICE       85.9947    240.3220      0
11   Return_Lag1            PRICE       -0.0801      0.1047      0
12   Return_Lag2            PRICE       -0.0801      0.1047      0
13   Gross_Marg

In [83]:
enriched['AAPL'].head(20).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,2020-07-31 00:00:00,2020-08-03 00:00:00,2020-08-04 00:00:00,2020-08-05 00:00:00,2020-08-06 00:00:00,2020-08-07 00:00:00,2020-08-10 00:00:00,2020-08-11 00:00:00,2020-08-12 00:00:00,2020-08-13 00:00:00,2020-08-14 00:00:00,2020-08-17 00:00:00,2020-08-18 00:00:00,2020-08-19 00:00:00,2020-08-20 00:00:00,2020-08-21 00:00:00,2020-08-24 00:00:00,2020-08-25 00:00:00,2020-08-26 00:00:00,2020-08-27 00:00:00
Open,102.88,108.2,109.13,109.38,110.41,113.2,112.6,111.97,110.5,114.43,114.83,116.06,114.35,115.98,115.75,119.26,128.7,124.7,126.18,127.14
High,106.42,111.64,110.79,110.39,114.41,113.67,113.78,112.48,113.28,116.04,115.0,116.09,116.0,117.16,118.39,124.87,128.78,125.18,126.99,127.48
Low,100.83,107.89,108.39,108.9,109.8,110.29,110.0,109.11,110.3,113.93,113.05,113.96,114.01,115.61,115.73,119.25,123.94,123.05,125.08,123.83
Close,106.26,108.94,109.67,110.06,113.9,111.11,112.73,109.38,113.01,115.01,114.91,114.61,115.56,115.71,118.28,124.37,125.86,124.83,126.52,125.01
Adj. Close,102.98,105.58,106.28,106.67,110.39,107.89,109.45,106.2,109.73,111.67,111.57,111.28,112.21,112.35,114.84,120.76,122.2,121.2,122.85,121.38
Volume,374295468,308151388,172792368,121991952,202428900,198045612,212403424,187902376,165944820,210082064,165565208,117725656,105633540,145538008,126907188,338054640,345937768,211495788,163022268,155552384
Dividend,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Shares Outstanding,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0,17102536000.0


## 5. Data Dictionary

### Group 1: Raw Price Columns *(reference only — not used as ML features)*

| Field | Type | Description |
|---|---|---|
| `Ticker` | string | Stock ticker symbol (e.g. `AAPL`) |
| `Date` | date | Trading day |
| `Open` | float | Opening price of the session |
| `High` | float | Highest price of the session |
| `Low` | float | Lowest price of the session |
| `Close` | float | Closing price (unadjusted) |
| `Adj. Close` | float | Closing price adjusted for splits and dividends — used for all price-based feature calculations in `etl.py` |
| `Volume` | int | Number of shares traded |
| `Dividend` | float | Dividend paid on that day (0 if none) |
| `Shares Outstanding` | float | Total shares outstanding |

---

### Group 2: Price-Based Engineered Features *(ML features)*

| Field | Type | Description |
|---|---|---|
| `Return` | float | Daily % change in `Adj. Close`. Captures how much the price moved today |
| `MA5` | float | 5-day simple moving average of `Adj. Close`. Smooths short-term noise (≈ 1 week trend) |
| `MA20` | float | 20-day simple moving average of `Adj. Close`. Captures medium-term trend (≈ 1 month) |
| `Volume_Change` | float | Daily % change in volume. Spikes signal unusual market activity |
| `Market_Cap` | float | Total market value = `Adj. Close × Shares Outstanding`. Dynamic proxy for company size — changes daily with price |
| `RSI` | float | Relative Strength Index (14-day). Momentum indicator: >70 = overbought, <30 = oversold |
| `MACD` | float | Difference between 12-day and 26-day EMA. Positive = bullish momentum, negative = bearish |
| `MACD_Signal` | float | 9-day EMA of MACD. Crossovers with MACD generate buy/sell signals |
| `BB_Upper` | float | Upper Bollinger Band (MA20 + 2×std). Price near here = overbought |
| `BB_Lower` | float | Lower Bollinger Band (MA20 − 2×std). Price near here = oversold |
| `Return_Lag1` | float | Yesterday's `Return`. Gives the model memory of the previous day |
| `Return_Lag2` | float | Return from 2 days ago. Captures short-term momentum patterns |

---

### Group 3: Fundamental Features *(ML features — quarterly, forward-filled)*

| Field | Type | Source file | Description |
|---|---|---|---|
| `Gross_Margin` | float | Income | Gross Profit / Revenue. How much revenue remains after production costs |
| `Operating_Margin` | float | Income | Operating Income / Revenue. Efficiency of core business operations |
| `Net_Margin` | float | Income | Net Income / Revenue. Bottom-line profitability after all expenses and taxes |
| `Debt_to_Equity` | float | Balance | Total Liabilities / Total Equity. Financial leverage — higher = more debt-dependent |
| `Operating_CF_Ratio` | float | Cashflow + Balance | Operating Cash Flow / Total Assets. Quality of earnings — harder to manipulate than net income |
| `Publish Date` | date | — | Date the quarterly report was published. Used for the point-in-time merge — not a feature |

---

### Group 4: Target *(what the model predicts)*

| Field | Type | Description |
|---|---|---|
| `Target` | float | **Next-day return**: `(price_tomorrow − price_today) / price_today`. Continuous regression target. Positive = price rises next day, negative = price falls. The web app converts this to a category and signal using fixed thresholds. |

**Category thresholds applied to predicted return:**

| Predicted return | Category | Trading signal |
|---|---|---|
| > +2.0% | HIGH RISE | BUY |
| +0.5% to +2.0% | LOW RISE | BUY |
| −0.5% to +0.5% | STAY | HOLD |
| −2.0% to −0.5% | LOW FALL | SELL |
| < −2.0% | HIGH FALL | SELL |

---

**Summary:** 17 trainable features (12 price + 5 fundamental) → 1 continuous regression target (next-day return).